In [ ]:

import sys
sys.path.insert(0, '..')
print("Python executable:", sys.executable)
print("Python version:", sys.version)
print("sys.path:", sys.path[:3])

# Try importing just constants
try:
    from petcr import constants
    print("constants module:", constants.__file__)
    print("KARMAN in constants:", hasattr(constants, 'KARMAN'))
    if hasattr(constants, 'KARMAN'):
        print("KARMAN value:", constants.KARMAN)
except Exception as e:
    print(f"Error importing constants: {e}")


# 地球在"出汗"？——什么是蒸散发？
# Earth is "Sweating"? - What is Evapotranspiration?

---

## 🌍 欢迎！Welcome!

**中文**：欢迎来到水文学的奇妙世界！这个教程将用最简单的方式，帮助你理解什么是**蒸散发（Evapotranspiration, ET）**。

**English**: Welcome to the wonderful world of hydrology! This tutorial will help you understand **Evapotranspiration (ET)** in the simplest way possible.

---

## 📖 从生活中的现象说起 | Starting from Everyday Phenomena

### 场景 1：湿毛巾在风中变干 | Scenario 1: A Wet Towel Drying in the Wind

想象你刚洗完澡，把湿毛巾挂在阳台上。什么时候毛巾会干得更快？

Imagine you just took a shower and hung a wet towel on the balcony. When does the towel dry faster?

1. **☀️ 阳光越强** → 温度越高 → 水分子运动越快 → 干得越快
   - **Stronger sunlight** → Higher temperature → Faster water molecule movement → Dries faster

2. **💨 风越大** → 空气流动越快 → 水蒸气被吹走 → 干得越快
   - **Stronger wind** → Faster air movement → Water vapor blown away → Dries faster

3. **☔ 空气越干燥**（湿度低）→ 空气"渴"了，能吸收更多水分 → 干得越快
   - **Drier air** (low humidity) → Air is "thirsty" and can absorb more moisture → Dries faster

**这就是蒸散发的物理基础！| This is the physical basis of evapotranspiration!**

---

## 🌱 地球表面也在"出汗" | Earth's Surface is Also "Sweating"

地球表面的水（河流、湖泊、土壤）和植物，每天都在向大气"出汗"——这就是**蒸散发**：

The water on Earth's surface (rivers, lakes, soil) and plants "sweat" into the atmosphere every day - this is **evapotranspiration**:

- **蒸发 (Evaporation)**：水面和土壤直接蒸发
  - Water evaporating directly from surfaces and soil
- **蒸腾 (Transpiration)**：植物通过叶片释放水蒸气
  - Plants releasing water vapor through leaves

**为什么重要？| Why is it important?**
- 🌧️ 影响降雨和气候变化
- 💧 决定水资源的可用性
- 🌾 影响农业灌溉需求

---

## 🧮 科学家如何计算蒸散发？| How Do Scientists Calculate ET?

### Penman 公式：能量项 + 动力项

1956年，英国科学家 Penman 提出了一个经典公式，把蒸散发分解为两部分：

In 1956, British scientist Penman proposed a classic formula that breaks ET into two parts:

$$
\text{ET} = \underbrace{\text{能量项 (Radiation Term)}}_\text{阳光提供的能量} + \underbrace{\text{动力项 (Aerodynamic Term)}}_\text{风和湿度的影响}
$$

**物理意义 | Physical Meaning**:

- **能量项**：太阳辐射提供的"烧开水"的能量
  - **Radiation term**: Energy from solar radiation to "boil water"
  
- **动力项**：风把水蒸气"吹走"的能力
  - **Aerodynamic term**: Wind's ability to "blow away" water vapor

---

## 💻 动手试试！| Let's Try It!

下面我们将用 **PET-CR** 库来计算蒸散发，并用**交互式滑块**来调节参数，看看不同条件下蒸散发如何变化。

Below we'll use the **PET-CR** library to calculate ET and use **interactive sliders** to adjust parameters and see how ET changes under different conditions.

In [ ]:

import sys
sys.path.insert(0, '..')  # Add parent directory to path

# Force reload of modules to clear cache
import importlib
if 'petcr' in sys.modules:
    # Remove all petcr modules from cache
    to_remove = [key for key in sys.modules if key.startswith('petcr')]
    for key in to_remove:
        del sys.modules[key]

import petcr

print("OK - Libraries imported successfully!")
print(f"PET-CR Version: {petcr.__version__ if hasattr(petcr, '__version__') else 'development'}")


## 🎨 交互式实验：改变温度和风速 | Interactive Experiment: Change Temperature and Wind Speed

**实验问题 | Experiment Question**:
- 温度升高10°C，蒸散发会增加多少？
- 风速加倍，蒸散发会怎么变化？

**提示 | Hint**: 拖动下面的滑块，观察图表变化！

In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import FloatSlider, interact

# Set up Chinese fonts
petcr.setup_chinese_font()

def explore_et_basics(temperature_c=20, wind_speed=2.0, relative_humidity=60):
    """
    交互式探索蒸散发的影响因素
    Interactive exploration of factors affecting ET
    
    Parameters:
    - temperature_c: 温度 (°C) | Temperature (°C)
    - wind_speed: 风速 (m/s) | Wind speed (m/s)  
    - relative_humidity: 相对湿度 (%) | Relative humidity (%)
    """
    
    # 假设一些标准气象条件 | Assume some standard meteorological conditions
    net_radiation = 200.0  # 净辐射 W/m2 | Net radiation W/m2
    air_pressure = 101325.0  # 气压 Pa | Air pressure Pa
    
    # 计算 Penman 潜在蒸散发 | Calculate Penman potential ET
    try:
        et_result = petcr.penman_potential_et(
            net_radiation=net_radiation,
            ground_heat_flux=0.0,
            temperature=temperature_c,
            relative_humidity=relative_humidity,
            wind_speed=wind_speed,
            pressure=air_pressure
        )
        et_w_m2 = et_result  # W/m2
        
        # 转换为 mm/day（常用单位）| Convert to mm/day (common unit)
        lambda_v = 2.45e6  # 潜热 J/kg | Latent heat J/kg
        et_mm_day = et_w_m2 * 86400 / lambda_v
        
    except Exception as e:
        print(f"计算出错 | Calculation error: {e}")
        et_w_m2 = 0
        et_mm_day = 0
    
    # 可视化结果 | Visualize results
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # 左图：能量收支 | Left: Energy budget
    ax1 = axes[0]
    energy_components = ['净辐射\nNet Radiation', '潜热通量\nLatent Heat', '剩余能量\nRemaining']
    energy_values = [net_radiation, et_w_m2, net_radiation - et_w_m2]
    colors = ['#FF6B6B', '#4ECDC4', '#95E1D3']
    
    bars = ax1.bar(energy_components, energy_values, color=colors, edgecolor='black', linewidth=1.5)
    ax1.set_ylabel('能量通量 (W/m2) | Energy Flux (W/m2)', fontsize=12, fontweight='bold')
    ax1.set_title('能量分配 | Energy Partitioning', fontsize=14, fontweight='bold')
    ax1.grid(axis='y', alpha=0.3, linestyle='--')
    
    # 添加数值标签 | Add value labels
    for bar, val in zip(bars, energy_values):
        height = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width()/2., height,
                f'{val:.1f}',
                ha='center', va='bottom', fontweight='bold')
    
    # 右图：关键信息卡片 | Right: Key information card
    ax2 = axes[1]
    ax2.axis('off')
    
    info_text = f"""
Results:

Temperature: {temperature_c:.1f} C
Wind Speed: {wind_speed:.1f} m/s
Relative Humidity: {relative_humidity:.0f}%

ET: {et_mm_day:.2f} mm/day ({et_w_m2:.1f} W/m2)

ET uses {(et_w_m2/net_radiation*100):.1f}% of net radiation
    """
    
    ax2.text(0.5, 0.5, info_text, 
             transform=ax2.transAxes,
             fontsize=11,
             verticalalignment='center',
             horizontalalignment='center',
             family='sans-serif',
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.3))
    
    plt.tight_layout()
    plt.show()

# Create interactive sliders
interact(explore_et_basics,
         temperature_c=FloatSlider(min=0, max=40, step=1, value=20, 
                                   description='Temperature (C):',
                                   style={'description_width': '120px'}),
         wind_speed=FloatSlider(min=0.5, max=10, step=0.5, value=2.0,
                               description='Wind Speed (m/s):',
                               style={'description_width': '120px'}),
         relative_humidity=FloatSlider(min=20, max=95, step=5, value=60,
                                      description='Humidity (%):',
                                      style={'description_width': '120px'})
);


## 🕐 一天中的蒸散发变化 | ET Variation Throughout the Day

太阳升起后，温度升高，蒸散发也会增加。让我们模拟一天24小时的变化：

After sunrise, as temperature increases, ET also rises. Let's simulate the variation over 24 hours:

In [ ]:

# Create figures directory if it doesn't exist
from pathlib import Path
Path('figures').mkdir(parents=True, exist_ok=True)

# 模拟一天的气象数据 | Simulate one day's meteorological data
hours = np.arange(0, 24, 1)  # 0-23点 | 0-23 hours

# 温度变化（正弦曲线模拟）| Temperature variation (simulated with sine curve)
temp_mean = 20  # 平均温度 | Mean temperature
temp_amplitude = 10  # 温度振幅 | Temperature amplitude
temperatures = temp_mean + temp_amplitude * np.sin(2 * np.pi * (hours - 6) / 24)

# 净辐射变化（白天高，夜晚低）| Net radiation (high during day, low at night)
net_radiation = 400 * np.maximum(0, np.sin(np.pi * hours / 24))

# 风速（假设恒定）| Wind speed (assumed constant)
wind_speed = 2.5  # m/s

# 相对湿度（早晨高，下午低）| Relative humidity (high in morning, low in afternoon)
rh = 80 - 30 * np.sin(np.pi * (hours - 6) / 24)

# 计算每小时的 ET | Calculate hourly ET
et_daily = []
air_pressure = 101325.0  # Pa

for i, hour in enumerate(hours):
    temp_c = temperatures[i]
    
    try:
        et_w = petcr.penman_potential_et(
            net_radiation=net_radiation[i],
            ground_heat_flux=0.0,
            temperature=temp_c,
            relative_humidity=rh[i],
            wind_speed=wind_speed,
            pressure=air_pressure
        )
        et_mm_hour = et_w * 3600 / 2.45e6  # 转换为 mm/hour
        et_daily.append(et_mm_hour)
    except:
        et_daily.append(0)

et_daily = np.array(et_daily)

# 绘制结果 | Plot results
fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)

# 温度和辐射 | Temperature and radiation
ax1 = axes[0]
ax1_twin = ax1.twinx()
line1 = ax1.plot(hours, temperatures, 'r-o', linewidth=2, label='温度 | Temperature', markersize=4)
line2 = ax1_twin.plot(hours, net_radiation, 'orange', linewidth=2, linestyle='--', 
                      label='净辐射 | Net Radiation', marker='s', markersize=4)
ax1.set_ylabel('温度 (°C) | Temperature (°C)', color='r', fontweight='bold', fontsize=11)
ax1_twin.set_ylabel('净辐射 (W/m2) | Net Radiation (W/m2)', color='orange', fontweight='bold', fontsize=11)
ax1.tick_params(axis='y', labelcolor='r')
ax1_twin.tick_params(axis='y', labelcolor='orange')
ax1.set_title('气象条件日变化 | Diurnal Variation of Meteorological Conditions', 
              fontsize=13, fontweight='bold')
ax1.grid(alpha=0.3)
lines = line1 + line2
labels = [l.get_label() for l in lines]
ax1.legend(lines, labels, loc='upper left')

# 相对湿度 | Relative humidity
ax2 = axes[1]
ax2.plot(hours, rh, 'b-^', linewidth=2, markersize=4)
ax2.fill_between(hours, rh, alpha=0.3, color='blue')
ax2.set_ylabel('相对湿度 (%) | Relative Humidity (%)', fontweight='bold', fontsize=11)
ax2.set_ylim([0, 100])
ax2.grid(alpha=0.3)

# 蒸散发 | Evapotranspiration
ax3 = axes[2]
ax3.plot(hours, et_daily, 'g-D', linewidth=2.5, markersize=5)
ax3.fill_between(hours, et_daily, alpha=0.4, color='green')
ax3.set_xlabel('时间 (小时) | Time (Hour)', fontweight='bold', fontsize=11)
ax3.set_ylabel('蒸散发 (mm/h) | ET (mm/h)', fontweight='bold', fontsize=11)
ax3.set_title('潜在蒸散发日变化 | Diurnal Variation of Potential ET', 
              fontsize=13, fontweight='bold')
ax3.grid(alpha=0.3)

# 添加每日总蒸散发 | Add daily total ET
total_et = np.sum(et_daily)
ax3.text(0.02, 0.95, f'日总蒸散发 | Daily Total ET: {total_et:.2f} mm',
         transform=ax3.transAxes,
         fontsize=11, fontweight='bold',
         bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.5),
         verticalalignment='top')

plt.tight_layout()
plt.savefig('figures/et_diurnal_variation.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nSimulation complete! Daily total ET: {total_et:.2f} mm")


## 🎯 核心要点总结 | Key Takeaways

通过这个教程，你学到了：

Through this tutorial, you learned:

### 1. 蒸散发的物理本质 | Physical Nature of ET
- 🌡️ **能量驱动**：温度越高，蒸发越快（就像烧水）
  - **Energy-driven**: Higher temperature → faster evaporation (like boiling water)
  
- 💨 **动力驱动**：风越大，水汽带走越快（就像吹湿毛巾）
  - **Aerodynamically-driven**: Stronger wind → faster moisture removal (like blowing on wet towel)

### 2. Penman 公式的两个组成部分 | Two Components of Penman Equation
$$
\text{ET}_{\text{Penman}} = \frac{\Delta}{\Delta + \gamma} R_n + \frac{\gamma}{\Delta + \gamma} E_a
$$

- $\Delta$: 饱和水汽压随温度的变化率（大气的"吸水能力"敏感度）
  - Rate of change of saturation vapor pressure with temperature (sensitivity of air's "water-absorbing ability")
  
- $\gamma$: 干湿表常数（能量和动力的权重）
  - Psychrometric constant (weight between energy and aerodynamic terms)
  
- $R_n$: 净辐射（太阳提供的能量）
  - Net radiation (energy from sun)
  
- $E_a$: 空气动力学项（风和湿度的影响）
  - Aerodynamic term (effect of wind and humidity)

### 3. 实际应用 | Practical Applications
- 🌾 **农业灌溉**：计算作物需要多少水
  - **Agricultural irrigation**: Calculate how much water crops need
  
- 🌧️ **气候预测**：理解水循环和降雨模式
  - **Climate prediction**: Understand water cycle and rainfall patterns
  
- 💧 **水资源管理**：评估区域水资源平衡
  - **Water resource management**: Assess regional water balance

---

## 📚 下一步 | Next Steps

恭喜你完成了第一个教程！接下来可以学习：

Congratulations on completing the first tutorial! Next, you can learn:

1. **Notebook 2**: 互补关系理论（为什么干旱时蒸发潜力反而增加？）
   - Complementary Relationship Theory (Why does evaporation potential increase during drought?)

2. **Notebook 3**: 归因分析（气候变暖 vs 植树造林，谁让水资源减少？）
   - Attribution Analysis (Climate warming vs afforestation: who reduces water resources?)

---

## 💡 思考题 | Discussion Questions

1. 为什么沙漠地区白天温度很高，但实际蒸发量可能不大？
   - Why do desert areas have high daytime temperatures but may not have large actual evaporation?

2. 全球变暖后，蒸散发会增加还是减少？为什么？
   - After global warming, will evapotranspiration increase or decrease? Why?

3. 如果在城市里多种树，会如何影响局地气候？
   - If we plant more trees in cities, how will it affect local climate?

---

**作者 | Author**: PET-CR Development Team  
**版本 | Version**: 1.0  
**日期 | Date**: 2025-12-04